# Thrail App Recommendation System (TARS 2.0) Master Mathematical Prototyping Scratchpad

This notebook contains the complete mathematical specifications, formulas, and code execution outputs for all core modules of **TARS 2.0** (`TARS_20260826.pdf`).

---

## 1. Core Module Setup & Imports
Imports internal TARS modules from `core/` (`distance_strategies`, `profile_manager`, `recommender`).

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from IPython.display import display

# Robust path resolution for recommendation_engine directory
curr_dir = os.getcwd()
if os.path.basename(curr_dir) == "benchmarks":
    REC_ENGINE_DIR = os.path.dirname(curr_dir)
elif os.path.exists(os.path.join(curr_dir, "core")):
    REC_ENGINE_DIR = curr_dir
elif os.path.exists(os.path.join(curr_dir, "recommendation_engine", "core")):
    REC_ENGINE_DIR = os.path.join(curr_dir, "recommendation_engine")
else:
    REC_ENGINE_DIR = curr_dir

if REC_ENGINE_DIR not in sys.path:
    sys.path.insert(0, REC_ENGINE_DIR)

from core.distance_strategies import DistanceStrategy, EngineRegistry, EnsembleDistanceStrategy, GowerDistanceStrategy, EuclideanDistanceStrategy, CosineDistanceStrategy
from core.profile_manager import build_17_feature_vector, build_18_feature_vector, TwoAnchorProfileManager, NUM_FEATURES
from core.recommender import HybridRecommender, calculate_alpha_tuner

## 2. 17-Feature Dataset Vector Mapping & Normalization Formulas ($p = 17$)

Maps raw trail profiles and user onboarding preferences into a normalized 17-dimensional vector $V \in [0, 1]^{17}$ (Duration dimension removed):

$$\text{Vector } V = [v_1, v_2, \dots, v_{17}]$$

### Feature Normalization Formulas:
1. **Provinces ($v_1 \dots v_5$)**: Cavite, Laguna, Batangas, Rizal, Quezon ($1.0$ if present, else $0.0$)
2. **Past Mountain Affinities ($v_6 \dots v_{10}$)**: Mt1 to Mt5 ($1.0$ if liked/hiked, else $0.0$)
3. **LASCO Rating Requirement ($v_{11}$)**:
   $$v_{11} = \frac{\text{lascoRating}}{9.0}$$
4. **Trail Length & Elevation Gain Index ($v_{12}$)**:
   $$\text{length\_norm} = \frac{\text{length}}{\max(\text{trail\_lengths})}, \quad \text{gain\_norm} = \frac{\text{gain}}{\max(\text{trail\_elevation\_gains})}$$
   $$v_{12} = \frac{\text{length\_norm} + \text{gain\_norm}}{\text{gain\_norm}}$$
5. **Tourism Infrastructure Flags ($v_{13} \dots v_{17}$)**: Infra 1 to 5 ($1.0$ if present, else $0.0$)

In [2]:
trails_path = os.path.join(REC_ENGINE_DIR, 'data', 'trails_mock.csv')
ratings_path = os.path.join(REC_ENGINE_DIR, 'data', 'user_ratings_mock.csv')

trails_df = pd.read_csv(trails_path)
ratings_df = pd.read_csv(ratings_path)

print(f"Loaded {len(trails_df)} candidate trails and {len(ratings_df)} historical user reviews.")

# Map Mt. Daraitan to a 17-feature vector
daraitan_dict = trails_df.iloc[0].to_dict()
daraitan_vec = build_17_feature_vector(daraitan_dict)
print("\nMt. Daraitan 17-Feature Vector Shape:", daraitan_vec.shape)
print("Vector:", daraitan_vec)

## 3. Distance Metrics & Engine Formulas

TARS utilizes three fundamental distance metrics between two vectors $a$ and $b$ ($p = 17$):

### a. Gower's Distance ($GD_{ab}$)
$$GD_{ab} = 1 - \frac{1}{p} \sum_{i=1}^p S_{ij}(a_{ij}, b_{ij})$$
where $S_{ij} = 1 - |a_i - b_i|$ for normalized continuous/ordinal features.

### b. Euclidean Distance ($ED_{ab}$)
$$ED_{ab} = \frac{\sqrt{\sum_{i=1}^p (a_i - b_i)^2}}{\sqrt{p}}$$

### c. Cosine Distance ($CD_{ab}$)
$$CD_{ab} = 1 - \frac{\sum_{i=1}^p a_i b_i}{\sqrt{\sum a_i^2} \sqrt{\sum b_i^2}}$$

In [3]:
vec_a = np.array([1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0.33, 0.20, 1, 1, 0, 0, 0], dtype=np.float32)
vec_b = np.array([1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0.50, 0.35, 1, 0, 0, 0, 0], dtype=np.float32)

gd_engine = GowerDistanceStrategy()
ed_engine = EuclideanDistanceStrategy()
cd_engine = CosineDistanceStrategy()

print(f"Gower's Distance GD(a, b):   {gd_engine.calculate_distance(vec_a, vec_b):.4f}")
print(f"Euclidean Distance ED(a, b): {ed_engine.calculate_distance(vec_a, vec_b):.4f}")
print(f"Cosine Distance CD(a, b):    {cd_engine.calculate_distance(vec_a, vec_b):.4f}")

## 4. Two-Anchor Profile Formulas ($P_e$ & $P_d$)

- **Easy Anchor Profile ($P_e$)**: Baseline predicted from user onboarding preferences survey.
- **Difficult Anchor Profile ($P_d$)**: Reverse-engineered upper-bound intensity boundary:
  $$P_d[10, 11] = \min(1.0, P_e[10, 11] + 0.35)$$

In [4]:
user_preferences = {
    'experience': 'Beginner',
    'province': ['Batangas', 'Rizal'],
    'hiked': True,
    'location': ['Mt. Batulao (Batangas)'],
    'tourism_1': True,
    'tourism_2': True
}

p_e, p_d = TwoAnchorProfileManager.create_anchor_profiles(user_preferences)
print("Easy Anchor Profile (P_e) difficulty dimensions [10, 11]:", p_e[10:12])
print("Difficult Anchor Profile (P_d) difficulty dimensions [10, 11]:", p_d[10:12])

## 5. Dynamic Alpha-Tuner Parameter Formula ($\alpha$)

Weight assigned to Content-Based vs Collaborative Filtering components based on active user count $u$ and steady-state threshold $m = 50$:

$$\alpha = \begin{cases} 1 - \frac{m}{2u} & \text{if } u \le m \\ 0.5 & \text{if } u > m \end{cases}$$

When $u = 0$ (cold start), $\alpha = 1.0$ (100% Content-Based). When $u = m$, $\alpha = 1 - \frac{m}{2m} = 0.50$ (balanced hybrid).

In [5]:
m = 50  # Steady state threshold
for u in [0, 10, 25, 50, 100]:
    alpha = calculate_alpha_tuner(u, m)
    print(f"Active Users u = {u:<4} -> Alpha Tuner alpha = {alpha:.4f}")

## 6. TARS Recommendation Score Equation ($R_{tu}$) & Score Interpretation Rules

$$\text{Recommendation Score } R_{tu} = \frac{\alpha (CB_{tu}) + (1 - \alpha) (CF_{tuv})}{\sqrt{17}}$$

where Collaborative Filtering score $CF_{tuv}$ between user $u$ and trail $t$ across peer users $v$ is:
$$CF_{tuv} = \frac{1}{m} \sum_{v=1}^m (CB_{uv})(CB_{vt})$$

### Score Interpretation Rules:
- $R_{tu} \rightarrow 0$: **Low uncertainty, high similarity match**
- $R_{tu} \rightarrow 1$: **High uncertainty, low similarity match**

In [6]:
recommender = HybridRecommender(trails_path, ratings_path, config_name='HYBRID_GOWER')

res = recommender.get_hybrid_recommendations(
    user_id='user_001',
    preferences=user_preferences,
    top_k=3,
    active_user_count=20
)

print(f"=== EASY ANCHOR PROFILE (P_e) RECOMMENDATIONS (Config: {res['config_used']}, Alpha: {res['alpha_tuner']}) ===")
display(pd.DataFrame(res['easy_anchor_recommendations']))

print(f"\n=== DIFFICULT ANCHOR PROFILE (P_d) RECOMMENDATIONS (Config: {res['config_used']}, Alpha: {res['alpha_tuner']}) ===")
display(pd.DataFrame(res['difficult_anchor_recommendations']))

## 7. Profile Update Function & Beta Corrector Factor ($\beta$) Matrix Formulas

When a user logs a completed hike $H_x$, the user profile $P_x$ ($P_e$ or $P_d$) updates as follows:

$$P_{xu} = P_{xo} + \frac{1}{k} \sum_{i=1}^k \beta_i (H_{xi} - P_{xo})$$

### Beta Corrector Factor (BCF) Matrix Formula:
$$\beta = \begin{cases} +\frac{R_{tu}}{2} & \text{if Matched (Actual == Predicted)} \\[6pt] -\frac{1 - R_{tu}}{2} & \text{if Mismatched (Actual != Predicted)} \end{cases}$$

#### Adjustment Matrix Truth Table:
| Actual Feedback | Predicted Anchor | BCF Multiplier (\beta) | Direction & Effect |
| :--- | :--- | :--- | :--- |
| **Easy** | **Easy** (Matched) | $+\frac{R_{tu}}{2}$ | Small bump towards $H_x$ (expands $P_e$ comfort zone) |
| **Difficult** | **Easy** (Mismatched) | $-\frac{1 - R_{tu}}{2}$ | Negative bump away from $H_x$ (pulls $P_e$ back) |
| **Easy** | **Difficult** (Mismatched) | $-\frac{1 - R_{tu}}{2}$ | Negative bump away from $H_x$ (shifts $P_d$ boundary) |
| **Difficult** | **Difficult** (Matched) | $+\frac{R_{tu}}{2}$ | Small bump towards $H_x$ (calibrates $P_d$ boundary) |

In [ ]:
r_tu_sample = 0.25  # Low uncertainty score
beta_matched = TwoAnchorProfileManager.compute_beta_corrector(r_tu_sample, is_matched=True)
beta_mismatched = TwoAnchorProfileManager.compute_beta_corrector(r_tu_sample, is_matched=False)

print(f"BCF Beta (Matched):    +{beta_matched:.4f}  (Positive bump towards hiked trail)")
print(f"BCF Beta (Mismatched): {beta_mismatched:.4f}  (Negative bump away from hiked trail)")

# Update Easy Profile P_e after an Easy hiked trail experience
p_e_updated = TwoAnchorProfileManager.update_profile(
    p_old=p_e,
    hiked_trails_vectors=[daraitan_vec],
    r_tu_scores=[r_tu_sample],
    actual_difficulties=["easy"],
    anchor_type="easy"
)

print("\nOriginal P_e (first 5 dims):", p_e[:5])
print("Updated  P_e (first 5 dims):", p_e_updated[:5])

## 8. ALL 23 BENCHMARK CONFIGURATIONS EVALUATION TABLE
Executes recommendations across **ALL 23 Benchmark Configurations** defined in `EngineRegistry`.

In [8]:
all_configs = EngineRegistry.BENCHMARK_CONFIGS
bench_results = []

for idx, (config_key, info) in enumerate(all_configs.items(), 1):
    res = recommender.get_hybrid_recommendations(
        user_id="user_scratchpad",
        preferences=user_preferences,
        top_k=1,
        active_user_count=20,
        config_override=config_key
    )
    top_easy = res["easy_anchor_recommendations"][0] if res["easy_anchor_recommendations"] else {"trail_name": "N/A", "r_tu": 1.0}
    top_diff = res["difficult_anchor_recommendations"][0] if res["difficult_anchor_recommendations"] else {"trail_name": "N/A", "r_tu": 1.0}
    
    bench_results.append({
        "#": idx,
        "Configuration Name": config_key,
        "Category": info["category"],
        "Description": info["desc"],
        "Top Easy Rec (P_e)": top_easy["trail_name"],
        "Easy R_tu": top_easy["r_tu"],
        "Top Diff Rec (P_d)": top_diff["trail_name"],
        "Diff R_tu": top_diff["r_tu"]
    })

bench_df = pd.DataFrame(bench_results)
print(f"=== ALL {len(bench_df)} BENCHMARK CONFIGURATIONS EVALUATION TABLE ===")
display(bench_df)